In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import Target, CouplingMap
from qiskit.quantum_info import Operator
from qiskit_device_benchmarking.bench_code.mrb import MirrorQA, QuantumAwesomeness
import pickle
import time

# Define parameters for the simulated backend
num_qubits = 5
basis_gates = ["id", "h", "x", "y", "z", "cx"]
p1 = 0.001  # 1-qubit gate error probability
p2 = 0.01   # 2-qubit gate error probability
rx_angle = np.pi / 2  # Match initial_entangling_angle

# Create a coupling map as a CouplingMap object
coupling_list = [[i, j] for i in range(num_qubits) for j in range(num_qubits) if i != j]
coupling_map = CouplingMap(couplinglist=coupling_list)

# Create a Target object to define the gates, including rz explicitly
target = Target.from_configuration(
    num_qubits=num_qubits,
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    custom_name_mapping={
        "id": Operator(np.array([[1, 0], [0, 1]])),  # Identity gate
        "h": Operator(np.array([[1, 1], [1, -1]]) / np.sqrt(2)),  # Hadamard gate
        "x": Operator(np.array([[0, 1], [1, 0]])),  # Pauli X gate
        "y": Operator(np.array([[0, -1j], [1j, 0]])),  # Pauli Y gate
        "z": Operator(np.array([[1, 0], [0, -1]])),  # Pauli Z gate
        "rx": Operator(np.array([[np.exp(-1j * rx_angle / 2), 0], [0, np.exp(1j * rx_angle / 2)]])),  # RZ gate for pi/2
        "cx": Operator(np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]]))  # CNOT gate
    }
)

# Create a noise model to emulate the NoisyBackend
noise_model = NoiseModel()

# Add depolarizing errors for 1-qubit and 2-qubit gates
error_1q = depolarizing_error(p1, 1)
error_2q = depolarizing_error(p2, 2)

# Apply errors to all basis gates except 'delay' and 'reset'
for gate in basis_gates:
    if gate in ["id", "h", "x", "y", "z", "rx"]:
        noise_model.add_all_qubit_quantum_error(error_1q, gate)
    elif gate == "cx":
        noise_model.add_all_qubit_quantum_error(error_2q, gate)

# Set up the AerSimulator with stabilizer method, target, and noise model
backend = AerSimulator(
    method="stabilizer",
    noise_model=noise_model if (p1 > 0 or p2 > 0) else None,
    target=target,
    max_parallel_threads=0,  # Use all available threads
    max_parallel_experiments=0
)

# Number of shots per circuit
shots = 10000

# Reduced lengths and samples for faster debugging
lengths = [2, 4, 10]
num_samples = 5

# Set up the experiment object
exp = MirrorQA(
    range(num_qubits),
    lengths=lengths,
    backend=backend,
    two_qubit_gate_density=0.25,
    num_samples=num_samples,
    initial_entangling_angle=np.pi/2,  # Clifford-compatible angle
)

# Set run options
exp.set_run_options(shots=shots)

# Run the experiment with timing
print("Starting simulation...")
start_time = time.time()
rb_data = exp.run()
end_time = time.time()
print(f"Simulation completed in {end_time - start_time:.2f} seconds")
print("Job IDs:", rb_data.job_ids)



Starting simulation...


TranspilerError: 'Unable to translate the operations in the circuit: ["rx"] to the backend\'s (or manually specified) target basis: {"box", "switch_case", "cx", "reset", "y", "measure", "while_loop", "delay", "if_else", "z", "h", "for_loop", "id", "x", "barrier", "snapshot", "store"}. This likely means the target basis is not universal or there are additional equivalence rules needed in the EquivalenceLibrary being used. For more details on this error see: https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes. BasisTranslator#translation-errors'

In [2]:
print('Pairs:', exp._pairs[0])
exp._static_trans_circuits[0].draw(fold=-1)

Pairs: [(3, 2)]


┌───┐ ░ ┌───┐            ░       ░ ┌───┐      ░ ┌───┐ ░  ░ ┌─┐            
   q_0: ┤ Z ├─░─┤ Z ├────────────░───────░─┤ Z ├──────░─┤ Y ├─░──░─┤M├────────────
        ├───┤ ░ ├───┤   ┌───┐    ░ ┌───┐ ░ ├───┤┌───┐ ░ ├───┤ ░  ░ └╥┘┌─┐         
   q_1: ┤ Y ├─░─┤ H ├───┤ Z ├────░─┤ Y ├─░─┤ H ├┤ X ├─░─┤ Y ├─░──░──╫─┤M├─────────
        ├───┤ ░ ├───┤   └───┘    ░ ├───┤ ░ ├───┤└───┘ ░ └───┘ ░  ░  ║ └╥┘┌─┐      
   q_2: ┤ Y ├─░─┤ X ├────────────░─┤ X ├─░─┤ X ├──────░───────░──░──╫──╫─┤M├──────
        ├───┤ ░ └─┬─┘┌─────────┐ ░ ├───┤ ░ └─┬─┘      ░ ┌───┐ ░  ░  ║  ║ └╥┘┌─┐   
   q_3: ┤ X ├─░───■──┤ Rx(π/2) ├─░─┤ Y ├─░───■────────░─┤ Z ├─░──░──╫──╫──╫─┤M├───
        └───┘ ░ ┌───┐└─────────┘ ░ ├───┤ ░ ┌───┐      ░ └───┘ ░  ░  ║  ║  ║ └╥┘┌─┐
   q_4: ──────░─┤ H ├────────────░─┤ Z ├─░─┤ H ├──────░───────░──░──╫──╫──╫──╫─┤M├
              ░ └───┘            ░ └───┘ ░ └───┘      ░       ░  ░  ║  ║  ║  ║ └╥┘
meas: 5/════════════════════════════════════════════════════════════╩══╩══╩══╩══╩═
                                                                    0  1  2  3  4

In [3]:
exp.analysis.set_options(analyzed_quantity='Effective Polarization')
#exp.analysis.set_options(analyzed_quantity='Mutual Information')
analysis = exp.analysis.run(rb_data)

In [4]:
print(rb_data.status())
print(rb_data.figure_names)


ExperimentStatus.ERROR
[]


In [5]:
analysis.figure(0)

ExperimentEntryNotFound: 'Figure index 0 out of range.'

In [33]:

# Debug: Inspect rb_data contents
print("Experiment data contents:")
try:
    data_entries = rb_data.data()
    for i, data in enumerate(data_entries):
        print(f"Data entry {i}: {data}")
        # Check required metadata
        required_keys = ["pairs", "singles", "target", "coupling_map"]
        metadata = data.get("metadata", {})
        for key in required_keys:
            print(f"  Metadata '{key}' present: {key in metadata}")
except Exception as e:
    print(f"Error accessing rb_data.data(): {e}")

# Save rb_data to a file for inspection
try:
    with open("rb_data.pkl", "wb") as f:
        pickle.dump(rb_data, f)
    print("Saved rb_data to rb_data.pkl for further inspection")
except Exception as e:
    print(f"Error saving rb_data: {e}")

# Try analysis with "Effective Polarization"
try:
    exp.analysis.set_options(analyzed_quantity="Effective Polarization")
    print("Running analysis for Effective Polarization...")
    start_time = time.time()
    analysis = exp.analysis.run(rb_data)
    end_time = time.time()
    print(f"Analysis for Effective Polarization completed in {end_time - start_time:.2f} seconds")
    print("Analysis results for Effective Polarization:",analysis.analysis_results(dataframe=True))
    print("Available figures for Effective Polarization:", analysis.figure_names)

    # Try to display the figure
    try:
        fig = analysis.figure(0)
        plt.show()
    except Exception as e:
        print(f"Error accessing figure for Effective Polarization: {e}")
except Exception as e:
    print(f"Error during analysis for Effective Polarization: {e}")

# Try analysis with "Mutual Information" as a fallback
try:
    exp.analysis.set_options(analyzed_quantity="Mutual Information")
    print("Running analysis for Mutual Information...")
    start_time = time.time()
    analysis_mi = exp.analysis.run(rb_data)
    end_time = time.time()
    print(f"Analysis for Mutual Information completed in {end_time - start_time:.2f} seconds")
    print("Analysis results for Mutual Information:", analysis.analysis_results(dataframe=True))
    print("Available figures for Mutual Information:", analysis_mi.figure_names)

    # Try to display the figure
    try:
        fig = analysis_mi.figure(0)
        plt.show()
    except Exception as e:
        print(f"Error accessing figure for Mutual Information: {e}")
except Exception as e:
    print(f"Error during analysis for Mutual Information: {e}")

# Fallback: Manual plotting if no figures are generated
try:
    if not analysis.figure_names and not analysis_mi.figure_names:
        print("No figures generated. Attempting manual plotting of results.")
        results = analysis.analysis_results() or analysis_mi.analysis_results()
        if results:
            lengths_data = []
            y_data = []
            for result in results:
                if hasattr(result, "value") and hasattr(result, "extra") and "length" in result.extra:
                    lengths_data.append(result.extra["length"])
                    y_data.append(result.value.n if hasattr(result.value, "n") else result.value)
        
            if lengths_data and y_data:
                plt.plot(lengths_data, y_data, 'o-', label="Analysis Results")
                plt.xlabel("Circuit Length")
                plt.ylabel("Value")
                plt.title("MirrorQA Analysis")
                plt.legend()
                plt.show()
            else:
                print("No valid data for manual plotting.")
        else:
            print("No analysis results available for plotting.")
except NameError:
    print("Analysis objects not defined due to earlier errors. Skipping manual plotting.")

Provider for ExperimentData object doesn't exist, resulting in a failed attempt to retrieve data from the server; no stored result data exists


Experiment data contents:
Saved rb_data to rb_data.pkl for further inspection
Running analysis for Effective Polarization...
Analysis for Effective Polarization completed in 0.00 seconds
Analysis results for Effective Polarization: Empty DataFrame
Columns: [name, experiment, components, value, quality, backend, run_time]
Index: []
Available figures for Effective Polarization: []
Error accessing figure for Effective Polarization: 'Figure index 0 out of range.'
Running analysis for Mutual Information...
Analysis for Mutual Information completed in 0.00 seconds
Analysis results for Mutual Information: Empty DataFrame
Columns: [name, experiment, components, value, quality, backend, run_time]
Index: []
Available figures for Mutual Information: []
Error accessing figure for Mutual Information: 'Figure index 0 out of range.'
No figures generated. Attempting manual plotting of results.
No analysis results available for plotting.


C:\Users\noname\AppData\Local\Temp\ipykernel_8820\4064996241.py:67: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  results = analysis.analysis_results() or analysis_mi.analysis_results()


In [17]:
ep_results = analysis.analysis_results(dataframe=True)
print("Type of results:", type(ep_results))
print(ep_results.head())  # Show first few rows if it's a DataFrame
mi_results = analysis_mi.analysis_results(dataframe=True)
print(mi_results.head())

Type of results: <class 'pandas.core.frame.DataFrame'>
Empty DataFrame
Columns: [name, experiment, components, value, quality, backend, run_time]
Index: []
Empty DataFrame
Columns: [name, experiment, components, value, quality, backend, run_time]
Index: []


In [20]:
# Debug: Inspect rb_data contents



from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd

pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
print("Experiment data contents:")
try:
    data_entries = rb_data.data()
    for i, data in enumerate(data_entries):
        print(f"Data entry {i}: {data}")
        # Check required metadata
        required_keys = ["pairs", "singles", "target", "coupling_map"]
        metadata = data.get("metadata", {})
        for key in required_keys:
            print(f"  Metadata '{key}' present: {key in metadata}")
except Exception as e:
    print(f"Error accessing rb_data.data(): {e}")

# Save rb_data to a file for inspection
try:
    with open("rb_data.pkl", "wb") as f:
        pickle.dump(rb_data, f)
    print("Saved rb_data to rb_data.pkl for further inspection")
except Exception as e:
    print(f"Error saving rb_data: {e}")

# Try analysis with "Effective Polarization"
try:
    exp.analysis.set_options(analyzed_quantity="Effective Polarization")
    print("Running analysis for Effective Polarization...")
    start_time = time.time()
    analysis = exp.analysis.run(rb_data)
    end_time = time.time()
    print(f"Analysis for Effective Polarization completed in {end_time - start_time:.2f} seconds")
    print("Analysis results for Effective Polarization:", analysis.analysis_results(dataframe=True))
    print("Available figures for Effective Polarization:", analysis.figure_names)

    # Try to display the figure
    try:
        fig = analysis.figure(0)
        plt.show()
    except Exception as e:
        print(f"Error accessing figure for Effective Polarization: {e}")

    # Extract results into DataFrame and plot
    import pandas as pd
    import matplotlib.pyplot as plt
    ep_results_df = analysis.analysis_results(dataframe=True)
    metrics = ['alpha', 'EPC', 'EI', 'chisq']
    values = [ep_results_df[ep_results_df['name'] == metric]['value'].iloc[0] for metric in metrics if not ep_results_df[ep_results_df['name'] == metric].empty]

    if values:
        plt.figure(figsize=(10, 6))
        plt.bar(metrics[:len(values)], values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
        plt.xlabel('Metrics')
        plt.ylabel('Value')
        plt.title('Effective Polarization Metrics')
        plt.yscale('log')  # Use log scale for better visibility of small values like EPC
        for i, v in enumerate(values):
            plt.text(i, v, f'{v:.2e}' if isinstance(v, float) else str(v), ha='center', va='bottom')
        plt.show()
    else:
        print("No valid metrics found for plotting.")

except Exception as e:
    print(f"Error during analysis for Effective Polarization: {e}")

# Try analysis with "Mutual Information" as a fallback

try:
    if not analysis.figure_names and not analysis_mi.figure_names:
        print("No figures generated. Attempting manual plotting of results.")
        results = analysis.analysis_results() or analysis_mi.analysis_results()
        if results:
            lengths_data = []
            y_data = []
            for result in results:
                if hasattr(result, "value") and hasattr(result, "extra") and "length" in result.extra:
                    lengths_data.append(result.extra["length"])
                    y_data.append(result.value.n if hasattr(result.value, "n") else result.value)
        
            if lengths_data and y_data:
                plt.plot(lengths_data, y_data, 'o-', label="Analysis Results")
                plt.xlabel("Circuit Length")
                plt.ylabel("Value")
                plt.title("MirrorQA Analysis")
                plt.legend()
                plt.show()
            else:
                print("No valid data for manual plotting.")
        else:
            print("No analysis results available for plotting.")
except NameError:
    print("Analysis objects not defined due to earlier errors. Skipping manual plotting.")

Provider for ExperimentData object doesn't exist, resulting in a failed attempt to retrieve data from the server; no stored result data exists


Experiment data contents:
Saved rb_data to rb_data.pkl for further inspection
Running analysis for Effective Polarization...
Analysis for Effective Polarization completed in 0.00 seconds
Analysis results for Effective Polarization: Empty DataFrame
Columns: [name, experiment, components, value, quality, backend, run_time]
Index: []
Available figures for Effective Polarization: []
Error accessing figure for Effective Polarization: 'Figure index 0 out of range.'
No valid metrics found for plotting.
No figures generated. Attempting manual plotting of results.
No analysis results available for plotting.


C:\Users\noname\AppData\Local\Temp\ipykernel_8820\4225873872.py:78: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  results = analysis.analysis_results() or analysis_mi.analysis_results()


In [21]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns

In [22]:
# Debug: Inspect rb_data contents
print("Experiment data contents:")
try:
    data_entries = rb_data.data()
    for i, data in enumerate(data_entries):
        print(f"Data entry {i}: {data}")
        # Check required metadata
        required_keys = ["pairs", "singles", "target", "coupling_map"]
        metadata = data.get("metadata", {})
        for key in required_keys:
            print(f"  Metadata '{key}' present: {key in metadata}")
except Exception as e:
    print(f"Error accessing rb_data.data(): {e}")

# Save rb_data to a file for inspection
try:
    with open("rb_data.pkl", "wb") as f:
        pickle.dump(rb_data, f)
    print("Saved rb_data to rb_data.pkl for further inspection")
except Exception as e:
    print(f"Error saving rb_data: {e}")

# Try analysis with "Effective Polarization"
try:
    exp.analysis.set_options(analyzed_quantity="Effective Polarization")
    print("Running analysis for Effective Polarization...")
    start_time = time.time()
    analysis = exp.analysis.run(rb_data)
    end_time = time.time()
    print(f"Analysis for Effective Polarization completed in {end_time - start_time:.2f} seconds")
    ep_results_df = analysis.analysis_results(dataframe=True)
    print("Analysis results for Effective Polarization:\n", ep_results_df)
    print("Available figures for Effective Polarization:", analysis.figure_names)

    # Try to display the figure
    try:
        fig = analysis.figure(0)
        plt.show()
    except Exception as e:
        print(f"Error accessing figure for Effective Polarization: {e}")

    # Extract results into DataFrame and plot
    import pandas as pd
    import matplotlib.pyplot as plt
    if not ep_results_df.empty:
        print("Debug: Available metrics in DataFrame:", ep_results_df['name'].unique())
        metrics = ['alpha', 'EPC', 'EI', 'chisq']
        values = []
        for metric in metrics:
            if (ep_results_df['name'] == metric).any():
                value = ep_results_df[ep_results_df['name'] == metric]['value'].iloc[0]
                if pd.isna(value) or value <= 0:
                    print(f"Warning: Invalid value for {metric}, skipping.")
                else:
                    values.append(float(value.n) if hasattr(value, 'n') else float(value))
            else:
                print(f"Metric {metric} not found in DataFrame.")
        
        if values:
            plt.figure(figsize=(10, 6))
            plt.bar([m for m, v in zip(metrics, values) if v is not None], values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(values)])
            plt.xlabel('Metrics')
            plt.ylabel('Value')
            plt.title('Effective Polarization Metrics')
            # plt.yscale('log')  # Comment out log scale for now, uncomment if needed
            for i, v in enumerate(values):
                plt.text(i, v, f'{v:.2e}' if v > 0 else str(v), ha='center', va='bottom')
            plt.show()
        else:
            print("No valid metrics found for plotting.")
    else:
        print("DataFrame is empty, no data to plot.")

except Exception as e:
    print(f"Error during analysis for Effective Polarization: {e}")

# Try analysis with "Mutual Information" as a fallback
try:
    exp.analysis.set_options(analyzed_quantity="Mutual Information")
    print("Running analysis for Mutual Information...")
    start_time = time.time()
    analysis_mi = exp.analysis.run(rb_data)
    end_time = time.time()
    print(f"Analysis for Mutual Information completed in {end_time - start_time:.2f} seconds")
    mi_results_df = analysis_mi.analysis_results(dataframe=True)
    print("Analysis results for Mutual Information:\n", mi_results_df)
    print("Available figures for Mutual Information:", analysis_mi.figure_names)

    # Try to display the figure
    try:
        fig = analysis_mi.figure(0)
        plt.show()
    except Exception as e:
        print(f"Error accessing figure for Mutual Information: {e}")

    # Extract results into DataFrame and plot
    if not mi_results_df.empty:
        print("Debug: Available metrics in DataFrame:", mi_results_df['name'].unique())
        metrics = ['alpha', 'EPC', 'EI', 'chisq']
        values = []
        for metric in metrics:
            if (mi_results_df['name'] == metric).any():
                value = mi_results_df[mi_results_df['name'] == metric]['value'].iloc[0]
                if pd.isna(value) or value <= 0:
                    print(f"Warning: Invalid value for {metric}, skipping.")
                else:
                    values.append(float(value.n) if hasattr(value, 'n') else float(value))
            else:
                print(f"Metric {metric} not found in DataFrame.")
        
        if values:
            plt.figure(figsize=(10, 6))
            plt.bar([m for m, v in zip(metrics, values) if v is not None], values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(values)])
            plt.xlabel('Metrics')
            plt.ylabel('Value')
            plt.title('Mutual Information Metrics')
            # plt.yscale('log')  # Comment out log scale for now, uncomment if needed
            for i, v in enumerate(values):
                plt.text(i, v, f'{v:.2e}' if v > 0 else str(v), ha='center', va='bottom')
            plt.show()
        else:
            print("No valid metrics found for plotting.")
    else:
        print("DataFrame is empty, no data to plot.")

except Exception as e:
    print(f"Error during analysis for Mutual Information: {e}")

# Fallback: Manual plotting if no figures are generated
try:
    if not analysis.figure_names and not analysis_mi.figure_names:
        print("No figures generated. Attempting manual plotting of results.")
        results = analysis.analysis_results() or analysis_mi.analysis_results()
        if results:
            lengths_data = []
            y_data = []
            for result in results:
                if hasattr(result, "value") and hasattr(result, "extra") and "length" in result.extra:
                    lengths_data.append(result.extra["length"])
                    y_data.append(result.value.n if hasattr(result.value, "n") else result.value)
        
            if lengths_data and y_data:
                plt.plot(lengths_data, y_data, 'o-', label="Analysis Results")
                plt.xlabel("Circuit Length")
                plt.ylabel("Value")
                plt.title("MirrorQA Analysis")
                plt.legend()
                plt.show()
            else:
                print("No valid data for manual plotting.")
        else:
            print("No analysis results available for plotting.")
except NameError:
    print("Analysis objects not defined due to earlier errors. Skipping manual plotting.")

Provider for ExperimentData object doesn't exist, resulting in a failed attempt to retrieve data from the server; no stored result data exists


Experiment data contents:
Saved rb_data to rb_data.pkl for further inspection
Running analysis for Effective Polarization...
Analysis for Effective Polarization completed in 0.00 seconds
Analysis results for Effective Polarization:
 Empty DataFrame
Columns: [name, experiment, components, value, quality, backend, run_time]
Index: []
Available figures for Effective Polarization: []
Error accessing figure for Effective Polarization: 'Figure index 0 out of range.'
DataFrame is empty, no data to plot.
Running analysis for Mutual Information...
Analysis for Mutual Information completed in 0.00 seconds
Analysis results for Mutual Information:
 Empty DataFrame
Columns: [name, experiment, components, value, quality, backend, run_time]
Index: []
Available figures for Mutual Information: []
Error accessing figure for Mutual Information: 'Figure index 0 out of range.'
DataFrame is empty, no data to plot.
No figures generated. Attempting manual plotting of results.
No analysis results available for

C:\Users\noname\AppData\Local\Temp\ipykernel_8820\525301957.py:133: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  results = analysis.analysis_results() or analysis_mi.analysis_results()


In [25]:
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit import QuantumCircuit
from qiskit.circuit.library import efficient_su2
 
n_qubits = 8
circuit = efficient_su2(n_qubits)



n_qubits = 500
circuit = efficient_su2(n_qubits)
measured_circuit = circuit.copy()
measured_circuit.measure_all()
 
rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)
 
# Initialize a Sampler backed by the stabilizer circuit simulator
exact_sampler = Sampler(
    options=dict(backend_options=dict(method="stabilizer"))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(
    1, AerSimulator(method="stabilizer")
)
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params)
job = exact_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()